### Streaming
Methods: stream() and astream() - used for synchronous and asynchronous streaming of results

### Stream Modes of Graph State

-  Values- Emits the full state of the graph after each node is called
- Updates: This streams only the updates happned to the state after the node is called 

In [ ]:
# Reducers define how updates to a state field are merged with the existing state. By default, updates overwrite previous values.
#  Reducers allow custom merge behavior such as appending messages, combining lists, aggregating results, or maintaining conversation history.
#  The most common reducer is add_messages, which appends new chat messages instead of replacing existing ones.

from typing import Annotated
from langgraph.graph.message import add_messages

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

class ChatState(TypedDict):
    messages: Annotated[list[str], add_messages]

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")




In [ ]:
from langchain_groq import ChatGroq

llm_groq = ChatGroq(model="qwen/qwen3-32b")
llm_groq.invoke("I'm John and like to play basketball")

In [ ]:
# Create Nodes

from langgraph.checkpoint.memory import MemorySaver

memory=MemorySaver()

config={"configuable":{"thread_id":"1"}}

def superbot(state:ChatState):
    return {"messages":[llm_groq.invoke(state["messages"], config)]}

In [ ]:
from IPython.display import Image, display


graph = StateGraph(ChatState)

# Adding nodes to the graph
graph.add_node("superbot", superbot)

# Adding edges to the graph
graph.add_edge(START, "superbot")
graph.add_edge("superbot", END)

graph_builder=graph.compile()

# Display the graph structure
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
# invoke the graph with an initial state
initial_state = {"messages": ["Hello, I'm John and like to play basketball"]}

graph_builder.invoke(initial_state, config)

### Streaming response with Stream Method

In [ ]:
# Stream the graph with an initial state
initial_state = {"messages": ["Hello, I'm John and like to play basketball"]}

for chunk in graph_builder.stream(initial_state, config, stream_mode="updates"): # only displays the recent AI message ( not the entire state is displayed)
    print(chunk)




In [ ]:
# Stream the graph with an initial state
initial_state = {"messages": ["Hello, I'm Dan and like to play Soccer"]}

for chunk in graph_builder.stream(initial_state, config, stream_mode="updates"): # displays the full state of the graph with both Human and AI meesage
    print(chunk)



### Streaming the response with astream 
- used when you need real time and non-blocking responses are needed
- Used when you want to stream more than the graph state .
- use astream_events methos, to stream back events which happen inside the node
- Each event is a dict with few keys
    - event: type 
    - name : name of the event
    - data : data associated with the event
    - metadata: contains node details emitting the event

In [ ]:
config ={"configurable":{
    "thread_id":"3"
}}

# astream_events() is an asynchronous LangGraph API that streams low-level execution events such as node starts, node completions, tool executions, and LLM invocations. 
# It is primarily used for debugging, tracing, monitoring, and understanding agent execution flow. 
# The thread_id in the config identifies the conversation whose state should be loaded and updated during execution.
async for event in graph_builder.astream_events(initial_state, config, version="v2"):
    print(event)